# SciEntsBank Baseline Grading Evaluation

Evaluate LLM grading accuracy on [SciEntsBank](https://huggingface.co/datasets/nkazi/SciEntsBank) using a simple baseline prompt.

- **Metric**: Quadratic Weighted Kappa (QWK), `weights="quadratic"`
- **Labels**: 0 = incorrect, 1 = partially correct, 2 = correct

## Notebook structure

| Section | Purpose |
|---------|----------|
| Cells 1–6 | Shared setup: deps, config, dataset, prompt, LLM interface, eval helpers |
| Cells 7+ | **One cell per model** — run independently, results stored in `RESULTS` |
| Last cell | Compare all completed runs |

To test a new model: add its HuggingFace ID to `MODEL_REGISTRY`, then duplicate a model cell.

In [1]:
# Cell 1: Install dependencies
!pip install -q datasets transformers accelerate scikit-learn tqdm huggingface_hub

In [2]:
# Cell 2: Global evaluation config (shared across all models)
from dataclasses import dataclass
from typing import Optional

@dataclass
class EvalConfig:
    eval_split: str = "test_ua"          # train | test_ua | test_uq | test_ud
    max_samples: Optional[int] = None    # None = full split; set e.g. 50 for quick test
    max_new_tokens: int = 3              # enough for a single digit label
    temperature: float = 0.0
    seed: int = 42

cfg = EvalConfig()

print(f"Eval split   : {cfg.eval_split}")
print(f"Max samples  : {cfg.max_samples or 'all'}")
print(f"Max tokens   : {cfg.max_new_tokens}")

Eval split   : test_ua
Max samples  : all
Max tokens   : 3


In [3]:
# Cell 3: Model registry — add new models here
MODEL_REGISTRY = {
    "qwen3-4b":       "Qwen/Qwen3-4B-Instruct-2507",
    "qwen2.5-3b":     "Qwen/Qwen2.5-3B-Instruct",
    "mistral-7b":     "mistralai/Mistral-7B-Instruct-v0.3",
    "llama-3.1-8b":   "meta-llama/Meta-Llama-3.1-8B-Instruct",
}

# Models that require HuggingFace login + license acceptance
GATED_MODELS = {"llama-3.1-8b"}

print("Registered models:")
for key, model_id in MODEL_REGISTRY.items():
    gated = " [gated]" if key in GATED_MODELS else ""
    print(f"  {key:16s} -> {model_id}{gated}")

Registered models:
  qwen3-4b         -> Qwen/Qwen3-4B-Instruct-2507
  qwen2.5-3b       -> Qwen/Qwen2.5-3B-Instruct
  mistral-7b       -> mistralai/Mistral-7B-Instruct-v0.3
  llama-3.1-8b     -> meta-llama/Meta-Llama-3.1-8B-Instruct [gated]


In [4]:
# Cell 4: HuggingFace login (required for gated models, e.g. Llama)
# Steps:
#   1. Accept license: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
#   2. Create token: https://huggingface.co/settings/tokens (Read access is enough)
#   3. In Colab: Secrets (🔑) -> add HF_TOKEN -> run this cell

import os
from huggingface_hub import login, get_token

hf_token = os.environ.get("HF_TOKEN")

# Colab Secrets
try:
    from google.colab import userdata
    hf_token = hf_token or userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace via HF_TOKEN.")
elif get_token() is not None:
    print("Already logged in to HuggingFace.")
else:
    print("No HF_TOKEN found. Paste your token when prompted:")
    login()
    print("Logged in to HuggingFace.")

Already logged in to HuggingFace.


In [5]:
# Cell 4: Load SciEntsBank and map labels to 0 / 1 / 2
from datasets import load_dataset

LABEL_MAP_5WAY = {
    "correct": 2,
    "partially_correct_incomplete": 1,
    "contradictory": 0,
    "irrelevant": 0,
    "non_domain": 0,
}

dataset = load_dataset("nkazi/SciEntsBank")
eval_ds = dataset[cfg.eval_split]

if cfg.max_samples is not None:
    eval_ds = eval_ds.select(range(min(cfg.max_samples, len(eval_ds))))

def get_gold_label(example: dict) -> int:
    label_name = example["label"]
    if isinstance(label_name, int):
        label_name = dataset[cfg.eval_split].features["label"].names[label_name]
    return LABEL_MAP_5WAY[label_name]

print(f"Loaded {len(eval_ds)} samples from split '{cfg.eval_split}'")
ex = eval_ds[0]
print(f"Example gold label: {get_gold_label(ex)}")

Loaded 540 samples from split 'test_ua'
Example gold label: 2


In [6]:
# Cell 5: Baseline grading prompt and response parser
import re

BASELINE_PROMPT = """You are a university professor for an introductory class.
Your job is to grade exercises and decide if the student answers are correct(2), partially correct(1), or incorrect(0).
Return ONLY a single digit: 0, 1, or 2. No explanation.

Question: {question}
Reference Answer: {reference_answer}
Student Answer: {student_answer}"""

def build_prompt(question: str, reference_answer: str, student_answer: str) -> str:
    return BASELINE_PROMPT.format(
        question=question,
        reference_answer=reference_answer,
        student_answer=student_answer,
    )

def parse_grade(response: str) -> int:
    """Extract integer label 0/1/2. Returns -1 on parse failure."""
    text = response.strip()
    if text in ("0", "1", "2"):
        return int(text)
    match = re.search(r"\b([012])\b", text)
    if match:
        return int(match.group(1))
    return -1

print("Prompt template and parser ready.")

Prompt template and parser ready.


In [7]:
# Cell 6: LLM interface + evaluation helpers (shared by all model cells)
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import get_token
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import cohen_kappa_score, accuracy_score, classification_report

RESULTS: dict[str, dict] = {}  # model_key -> metrics dict


class LLMGrader:
    """Unified grader interface. Instantiate per model, then unload to free GPU."""

    def __init__(self, model_id: str, max_new_tokens: int = 3, temperature: float = 0.0):
        self.model_id = model_id
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        hf_token = get_token()
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, trust_remote_code=True, token=hf_token
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
            trust_remote_code=True,
            token=hf_token,
        )
        self.model.eval()

    def grade(self, question: str, reference_answer: str, student_answer: str) -> tuple[int, str]:
        user_content = build_prompt(question, reference_answer, student_answer)
        messages = [{"role": "user", "content": user_content}]

        if hasattr(self.tokenizer, "apply_chat_template") and self.tokenizer.chat_template:
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        else:
            prompt = user_content

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=self.temperature > 0,
                temperature=self.temperature if self.temperature > 0 else None,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        raw = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        return parse_grade(raw), raw


def unload_grader(grader: LLMGrader) -> None:
    """Free GPU memory before loading the next model."""
    del grader.model
    del grader.tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def compute_metrics(y_true, y_pred, records, model_key: str, model_id: str) -> dict:
    valid_idx = [i for i, p in enumerate(y_pred) if p in (0, 1, 2)]
    y_true_v = [y_true[i] for i in valid_idx]
    y_pred_v = [y_pred[i] for i in valid_idx]

    qwk = cohen_kappa_score(y_true_v, y_pred_v, weights="quadratic")
    acc = accuracy_score(y_true_v, y_pred_v)
    parse_failures = sum(1 for p in y_pred if p == -1)

    return {
        "model_key": model_key,
        "model_id": model_id,
        "split": cfg.eval_split,
        "total": len(y_true),
        "valid": len(y_true_v),
        "parse_failures": parse_failures,
        "qwk": qwk,
        "accuracy": acc,
        "y_true": y_true,
        "y_pred": y_pred,
        "records": records,
    }


def print_metrics(result: dict) -> None:
    print("=" * 50)
    print(f"Model          : {result['model_key']} ({result['model_id']})")
    print(f"Split          : {result['split']}")
    print(f"Samples        : {result['valid']} valid / {result['total']} total")
    print(f"Parse failures : {result['parse_failures']}")
    print(f"QWK            : {result['qwk']:.4f}")
    print(f"Accuracy       : {result['accuracy']:.4f}")
    print("=" * 50)

    valid_idx = [i for i, p in enumerate(result["y_pred"]) if p in (0, 1, 2)]
    y_true_v = [result["y_true"][i] for i in valid_idx]
    y_pred_v = [result["y_pred"][i] for i in valid_idx]
    print(classification_report(
        y_true_v, y_pred_v,
        labels=[0, 1, 2],
        target_names=["incorrect (0)", "partial (1)", "correct (2)"],
        zero_division=0,
    ))


def run_evaluation(model_key: str) -> dict:
    """Load model, run full eval, store result in RESULTS, free GPU."""
    assert model_key in MODEL_REGISTRY, f"Unknown model key: {model_key}"
    model_id = MODEL_REGISTRY[model_key]

    if model_key in GATED_MODELS and get_token() is None:
        raise RuntimeError(
            f"'{model_key}' is a gated model. Before running:\n"
            f"  1. Accept license: https://huggingface.co/{model_id}\n"
            f"  2. Run Cell 4 (HuggingFace login) with a valid HF_TOKEN"
        )

    print(f"Loading {model_key} ({model_id}) ...")
    grader = LLMGrader(model_id, max_new_tokens=cfg.max_new_tokens, temperature=cfg.temperature)

    y_true, y_pred, records = [], [], []
    for example in tqdm(eval_ds, desc=f"Grading [{model_key}]"):
        gold = get_gold_label(example)
        pred, raw = grader.grade(
            example["question"],
            example["reference_answer"],
            example["student_answer"],
        )
        y_true.append(gold)
        y_pred.append(pred)
        records.append({
            "id": example["id"],
            "gold": gold,
            "pred": pred,
            "raw_response": raw,
            "question": example["question"],
            "reference_answer": example["reference_answer"],
            "student_answer": example["student_answer"],
        })

    unload_grader(grader)
    result = compute_metrics(y_true, y_pred, records, model_key, model_id)
    RESULTS[model_key] = result
    print_metrics(result)
    return result


print("Evaluation helpers ready. Call run_evaluation('model_key') in the cells below.")

Evaluation helpers ready. Call run_evaluation('model_key') in the cells below.


---
## Model evaluations

Run **one cell at a time** (each loads a full model into GPU).
Results are saved to the global `RESULTS` dict.

In [8]:
# Cell 7: Evaluate Qwen3-4B-Instruct
# result_qwen3_4b = run_evaluation("qwen3-4b")

In [9]:
# Cell 8: Evaluate Llama-3.1-8B-Instruct (gated — run Cell 4 first)
# 1. Accept license: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
# 2. Add HF_TOKEN to Colab Secrets, then run Cell 4
# 3. Run this cell

from huggingface_hub import get_token

assert get_token() is not None, "Run Cell 4 (HuggingFace login) first."
result_llama_3_1_8b = run_evaluation("llama-3.1-8b")

Loading llama-3.1-8b (meta-llama/Meta-Llama-3.1-8B-Instruct) ...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct.
403 Client Error. (Request ID: Root=1-6a58459d-6c774fab7885529d743a0eca;cd4947a9-3f39-4860-80ce-70372dca0bc2)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct to ask for access.

### Other models

Uncomment and run the cells below as needed. Each calls the same `run_evaluation()` interface.

In [10]:
# Cell 9: Evaluate Qwen2.5-3B-Instruct
result_qwen2_5_3b = run_evaluation("qwen2.5-3b")

Loading qwen2.5-3b (Qwen/Qwen2.5-3B-Instruct) ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Grading [qwen2.5-3b]:   0%|          | 0/540 [00:00<?, ?it/s]

Model          : qwen2.5-3b (Qwen/Qwen2.5-3B-Instruct)
Split          : test_ua
Samples        : 540 valid / 540 total
Parse failures : 0
QWK            : 0.5529
Accuracy       : 0.6222
               precision    recall  f1-score   support

incorrect (0)       0.59      0.91      0.72       194
  partial (1)       0.30      0.13      0.18       113
  correct (2)       0.75      0.62      0.68       233

     accuracy                           0.62       540
    macro avg       0.55      0.55      0.53       540
 weighted avg       0.60      0.62      0.59       540



In [11]:
# Cell 10: Evaluate Mistral-7B-Instruct
result_mistral_7b = run_evaluation("mistral-7b")

Loading mistral-7b (mistralai/Mistral-7B-Instruct-v0.3) ...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Grading [mistral-7b]:   0%|          | 0/540 [00:00<?, ?it/s]

Model          : mistral-7b (mistralai/Mistral-7B-Instruct-v0.3)
Split          : test_ua
Samples        : 540 valid / 540 total
Parse failures : 0
QWK            : 0.3264
Accuracy       : 0.5352
               precision    recall  f1-score   support

incorrect (0)       0.70      0.41      0.52       194
  partial (1)       0.18      0.05      0.08       113
  correct (2)       0.52      0.87      0.65       233

     accuracy                           0.54       540
    macro avg       0.47      0.45      0.42       540
 weighted avg       0.51      0.54      0.48       540



In [ ]:
# Cell 11: Add a new model
# 1. Add entry to MODEL_REGISTRY in Cell 3
# 2. Set NEW_MODEL_KEY below and run this cell
NEW_MODEL_KEY = "your-model-key"  # <-- replace with key from MODEL_REGISTRY

if NEW_MODEL_KEY in MODEL_REGISTRY:
    result_new_model = run_evaluation(NEW_MODEL_KEY)
else:
    print(f"Set NEW_MODEL_KEY to a registered key. Available: {list(MODEL_REGISTRY.keys())}")

In [ ]:
# Cell 12: Compare all completed runs
if not RESULTS:
    print("No results yet. Run at least one model evaluation cell above.")
else:
    summary = pd.DataFrame([
        {
            "model_key": r["model_key"],
            "model_id": r["model_id"],
            "split": r["split"],
            "valid/total": f"{r['valid']}/{r['total']}",
            "parse_failures": r["parse_failures"],
            "QWK": round(r["qwk"], 4),
            "Accuracy": round(r["accuracy"], 4),
        }
        for r in RESULTS.values()
    ]).sort_values("QWK", ascending=False)

    display(summary)

    # Preview misclassified examples for the best model
    best_key = summary.iloc[0]["model_key"]
    df = pd.DataFrame(RESULTS[best_key]["records"])
    df_valid = df[df["pred"].isin([0, 1, 2])].copy()
    df_valid["correct"] = df_valid["gold"] == df_valid["pred"]
    print(f"\nMisclassified examples for best model [{best_key}] ({(~df_valid['correct']).sum()}):")
    display(df_valid[~df_valid["correct"]].head(10))